# Imports and loading the dataset

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
pd.set_option("display.max_columns",50)
PROJECT_ROOT = Path("..")
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "issued-building-permits.xlsx"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

In [3]:
#Loading the raw data

In [4]:
df = pd.read_excel(RAW_FILE)
print(df.shape)
df.head(3)

(52199, 20)


,PermitNumber,PermitNumberCreatedDate,IssueDate,PermitElapsedDays,ProjectValue,TypeOfWork,Address,ProjectDescription,PermitCategory,Applicant,ApplicantAddress,PropertyUse,SpecificUseCategory,BuildingContractor,BuildingContractorAddress,IssueYear,GeoLocalArea,Geom,YearMonth,geo_point_2d
0,BP-2023-00680,2023-02-23,2023-04-17,53,150000.0,Addition / Alteration,"778 ROBSON STREET, Vancouver, BC V6Z 1A1",High Density Housing / Commercial - Addition /...,NaN,Tracy Tsang,"1662 75th Ave W\nVancouver, BC V6P6G2",Service Uses,Restaurant - Class 1,NaN,NaN,2023,Downtown,"{""coordinates"":[-123.1209527,49.2815546],""type...",2023-04,"49.2815546, -123.1209527"
1,DB-2024-00809,2024-03-07,2024-04-05,29,250000.0,Addition / Alteration,"155 E 24TH AVENUE, Vancouver, BC V5V 1Z5",Enquiry Centre - Addition / Alteration - Scope...,Renovation - Residential - Lower Complexity,JULIE TAN DBA: GREATVIEW DESIGN & DEVELOPMENT INC,"923-409 GRANVILLE STREET\nVANCOUVER, BC V6C 1T2",Dwelling Uses,Single Detached House w/Sec Suite,NaN,NaN,2024,Riley Park,"{""coordinates"":[-123.1022021,49.249743],""type""...",2024-04,"49.249743, -123.1022021"
2,DB-2021-04881,2021-09-16,2023-04-18,579,30000.0,Addition / Alteration,"4431 BLENHEIM STREET, Vancouver, BC V6L 2Z8",Low Density Housing - Addition / Alteration - ...,NaN,JULIE TAN DBA: GREATVIEW DESIGN & DEVELOPMENT INC,"923-409 GRANVILLE STREET\nVANCOUVER, BC V6C 1T2",Dwelling Uses,Single Detached House,NaN,NaN,2023,Dunbar-Southlands,"{""coordinates"":[-123.1789879,49.2467792],""type...",2023-04,"49.2467792, -123.1789879"


In [5]:
#Looking at structure and quality

In [6]:
print(df.dtypes)
print()
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate PermitNumber:", df["PermitNumber"].duplicated().sum())

missing = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing[missing > 0]

PermitNumber                     str
PermitNumberCreatedDate          str
IssueDate                        str
PermitElapsedDays              int64
ProjectValue                 float64
TypeOfWork                       str
Address                          str
ProjectDescription               str
PermitCategory                   str
Applicant                        str
ApplicantAddress                 str
PropertyUse                      str
SpecificUseCategory              str
BuildingContractor               str
BuildingContractorAddress        str
IssueYear                      int64
GeoLocalArea                     str
Geom                             str
YearMonth                        str
geo_point_2d                     str
dtype: object

Duplicate rows: 0
Duplicate PermitNumber: 0


BuildingContractorAddress    57.6
PermitCategory               42.6
BuildingContractor           38.0
GeoLocalArea                  0.8
Geom                          0.7
geo_point_2d                  0.7
Address                       0.4
ApplicantAddress              0.3
dtype: float64

In [7]:
cleaning_log = []

def log_step(name, data, note=""):
    cleaning_log.append({"step": name, "rows": len(data), "cols": data.shape[1], "note": note})
    print(f"{name:45s} rows={len(data):>7,}  cols={data.shape[1]:>3}  {note}")

log_step("Load raw data", df)

Load raw data                                 rows= 52,199  cols= 20  


# Data Cleaning

In [8]:
# Removing duplicates(just to be safe even though none exist)

In [9]:
before = len(df)
df = df.drop_duplicates().drop_duplicates(subset="PermitNumber", keep="first")
log_step("Drop duplicate rows / permit numbers", df, f"removed {before - len(df)}")

Drop duplicate rows / permit numbers          rows= 52,199  cols= 20  removed 0


In [10]:
# Parsing the date columns

In [11]:
df["created"] = pd.to_datetime(df["PermitNumberCreatedDate"], errors="coerce")
df["issued"] = pd.to_datetime(df["IssueDate"], errors="coerce")

print("Unparseable dates:", df[["created", "issued"]].isna().sum().to_dict())
print("Created range:", df["created"].min().date(), "to", df["created"].max().date())

df = df.dropna(subset=["created", "issued"])
log_step("Parse dates, drop unparseable", df)

Unparseable dates: {'created': 0, 'issued': 0}
Created range: 2013-07-25 to 2026-09-28
Parse dates, drop unparseable                 rows= 52,199  cols= 22  


In [12]:
# Checking the processsing-time column

In [13]:
print(df["PermitElapsedDays"].describe().round(1))
print("Zero-day permits:", (df["PermitElapsedDays"] == 0).sum())
print("Negative values:", (df["PermitElapsedDays"] < 0).sum())

# Sanity check: does issued - created reproduce PermitElapsedDays?
calc = (df["issued"] - df["created"]).dt.days
print("Match with issued - created:", ((calc - df["PermitElapsedDays"]).abs() <= 1).mean().round(3))

df = df[df["PermitElapsedDays"] >= 0]
log_step("Remove negative processing days", df)

count    52199.0
mean       134.5
std        149.5
min          0.0
25%         36.0
50%         85.0
75%        186.0
max       2807.0
Name: PermitElapsedDays, dtype: float64
Zero-day permits: 225
Negative values: 0
Match with issued - created: 1.0
Remove negative processing days               rows= 52,199  cols= 22  


In [14]:
# Building the target

In [15]:
median_days = df["PermitElapsedDays"].median()
df["on_time"] = (df["PermitElapsedDays"] <= median_days).astype(int)

print("Median processing days:", median_days)
print(df["on_time"].value_counts(normalize=True).round(3))
log_step("Build target on_time", df, f"median = {median_days:.0f} days; on-time share = {df.on_time.mean():.3f}")

Median processing days: 85.0
on_time
1    0.502
0    0.498
Name: proportion, dtype: float64
Build target on_time                          rows= 52,199  cols= 23  median = 85 days; on-time share = 0.502


In [16]:
# Project Value(Zeroes and Extreme Skew)

In [17]:
print(df["ProjectValue"].describe().round(0))
print("Zero-value share:", round((df["ProjectValue"] <= 0).mean(), 3))
print(df["ProjectValue"].quantile([.5, .9, .99, .999]))

df["value_is_zero"] = (df["ProjectValue"] <= 0).astype(int)
df["log_value"] = np.log1p(df["ProjectValue"].clip(lower=0))

print(df.groupby("value_is_zero")["on_time"].mean().round(3))
log_step("Project value: zero flag + log transform", df, f"zero share = {df.value_is_zero.mean():.3f}")

count    5.219900e+04
mean     7.995170e+05
std      1.052097e+07
min      0.000000e+00
25%      1.500000e+04
50%      5.000000e+04
75%      2.501000e+05
max      1.468680e+09
Name: ProjectValue, dtype: float64
Zero-value share: 0.152
0.500       50000.0
0.900      852872.0
0.990    11503000.0
0.999    95109072.0
Name: ProjectValue, dtype: float64
value_is_zero
0    0.479
1    0.636
Name: on_time, dtype: float64
Project value: zero flag + log transform      rows= 52,199  cols= 25  zero share = 0.152


In [18]:
# Handling Missing Values

In [19]:
df["PermitCategory"] = df["PermitCategory"].fillna("Unknown")
df["GeoLocalArea"] = df["GeoLocalArea"].fillna("Unknown")
df["PropertyUse"] = df["PropertyUse"].fillna("Unknown")

# Contractor name is 38% missing: keep only if it was listed
df["has_contractor"] = df["BuildingContractor"].notna().astype(int)

print(df.groupby("has_contractor")["on_time"].mean().round(3))
log_step("Fill missing category/area/use; contractor flag", df)

has_contractor
0    0.622
1    0.429
Name: on_time, dtype: float64
Fill missing category/area/use; contractor flag rows= 52,199  cols= 26  


In [20]:
# Grouping rare categories

In [21]:
print("PermitCategory values:", df["PermitCategory"].nunique())
print("PropertyUse values:", df["PropertyUse"].nunique())

TOP_USES = 20
top_uses = df["PropertyUse"].value_counts().head(TOP_USES).index
print("Share of rows covered by top 20 uses:", round(df["PropertyUse"].isin(top_uses).mean(), 3))

df["property_use"] = df["PropertyUse"].where(df["PropertyUse"].isin(top_uses), "Other")
df["permit_category"] = df["PermitCategory"]   # only 6 values, no grouping needed

log_step("Group rare PropertyUse values", df, f"{df['property_use'].nunique()} levels kept")

PermitCategory values: 5
PropertyUse values: 154
Share of rows covered by top 20 uses: 0.986
Group rare PropertyUse values                 rows= 52,199  cols= 28  21 levels kept


In [22]:
# Application time features and coordinates

In [23]:
df["created_year"] = df["created"].dt.year
df["created_month"] = df["created"].dt.month

coords = df["geo_point_2d"].str.split(",", expand=True)
df["lat"] = pd.to_numeric(coords[0], errors="coerce")
df["lon"] = pd.to_numeric(coords[1], errors="coerce")

print("Rows without coordinates:", df["lat"].isna().sum())
print(df.groupby("created_year")["on_time"].agg(["size", "mean"]).round(3))
log_step("Time features + coordinates", df)

Rows without coordinates: 362
              size   mean
created_year             
2013             1  0.000
2014             8  0.000
2015            46  0.000
2016          2728  0.136
2017          6307  0.545
2018          6321  0.527
2019          5553  0.548
2020          3908  0.509
2021          6478  0.385
2022          5102  0.452
2023          4248  0.520
2024          4845  0.558
2025          4676  0.592
2026          1978  0.799
Time features + coordinates                   rows= 52,199  cols= 32  


### Restricting to permits created 2017-2025

The dataset contains only permits *issued* from 2017 onward and was extracted in September 2026.
This creates two sampling biases in the processing-time target:

- **Left truncation (created 2013-2016, 2,783 rows):** a permit created before 2017 appears only
  if it took until 2017 or later to issue, so these permits are almost all "delayed" (0-14% on time).
  Fast permits from those years are missing.
- **Right censoring (created 2026, 1,978 rows):** permits created in 2026 appear only if already
  issued, so still-pending (slow) permits are missing and the on-time rate is inflated (79.9%).

Keeping these rows would let the model learn the data window instead of real permit behaviour.
We are therefore keeping permits created from 2017-2025 (47,438 rows, about 91% of the data).

In [24]:
# Keeping permits created 2017-2025 because older and 2026 permits are only 
# partly recorded, which would bias the on-time target.

In [25]:
before = len(df)
df = df[(df["created_year"] >= 2017) & (df["created_year"] <= 2025)].copy()

print("Rows removed:", before - len(df))
print(df["on_time"].value_counts(normalize=True).round(3))
log_step("Keep permits created 2017-2025", df, "removed pre-2017 and 2026")

Rows removed: 4761
on_time
1    0.512
0    0.488
Name: proportion, dtype: float64
Keep permits created 2017-2025                rows= 47,438  cols= 32  removed pre-2017 and 2026


In [26]:
# keeping only the columns we need

In [27]:
final = df[[
    "TypeOfWork", "property_use", "GeoLocalArea", "permit_category",
    "log_value", "ProjectValue", "value_is_zero", "has_contractor",
    "created_year", "created_month", "lat", "lon",
    "PermitElapsedDays",      # kept for EDA / cluster profiling ONLY, never a model feature
    "on_time",                # target
]].rename(columns={
    "TypeOfWork": "type_of_work",
    "GeoLocalArea": "area",
    "ProjectValue": "project_value",
    "PermitElapsedDays": "elapsed_days",
})

print(final.shape)
final.head()

(47438, 14)


,type_of_work,property_use,area,permit_category,log_value,project_value,value_is_zero,has_contractor,created_year,created_month,lat,lon,elapsed_days,on_time
0,Addition / Alteration,Service Uses,Downtown,Unknown,11.918397,150000.0,0,0,2023,2,49.281555,-123.120953,53,1
1,Addition / Alteration,Dwelling Uses,Riley Park,Renovation - Residential - Lower Complexity,12.429220,250000.0,0,0,2024,3,49.249743,-123.102202,29,1
2,Addition / Alteration,Dwelling Uses,Dunbar-Southlands,Unknown,10.308986,30000.0,0,0,2021,9,49.246779,-123.178988,579,0
3,Addition / Alteration,Retail Uses,Downtown,Renovation - Commercial/ Mixed Use - Lower Com...,12.429220,250000.0,0,0,2023,11,49.283397,-123.117746,31,1
5,Addition / Alteration,Office Uses,Downtown,Renovation - Commercial/ Mixed Use - Lower Com...,11.849405,140000.0,0,1,2024,1,49.287709,-123.118014,8,1


In [28]:
# Final quality checks

In [29]:
print("Missing values:")
print(final.isna().sum()[final.isna().sum() > 0])
print()
print(final.dtypes)
print()
print(final["type_of_work"].value_counts())
print()
print("Areas:", final["area"].nunique(), "| Property uses:", final["property_use"].nunique())
print(final.groupby("type_of_work")["on_time"].mean().round(2))

Missing values:
lat    348
lon    348
dtype: int64

type_of_work           str
property_use           str
area                   str
permit_category        str
log_value          float64
project_value      float64
value_is_zero        int64
has_contractor       int64
created_year         int32
created_month        int32
lat                float64
lon                float64
elapsed_days         int64
on_time              int64
dtype: object

type_of_work
Addition / Alteration                   23813
New Building                            10510
Salvage and Abatement                    6328
Demolition / Deconstruction              6267
Temporary Building / Structure            461
Outdoor Uses (No Buildings Proposed)       59
Name: count, dtype: int64

Areas: 23 | Property uses: 21
type_of_work
Addition / Alteration                   0.74
Demolition / Deconstruction             0.13
New Building                            0.08
Outdoor Uses (No Buildings Proposed)    0.66
Salvage and Abat

In [30]:
final.to_csv(CLEAN_DIR / "permits_clean.csv", index=False)
log_step("Keep only required columns", final, "dropped identifiers, text, geometry columns, IssueDate and other leakage columns; kept 14 columns")
pd.DataFrame(cleaning_log).to_csv(CLEAN_DIR / "cleaning_log.csv", index=False)
check = pd.read_csv(CLEAN_DIR / "permits_clean.csv")
print("Saved:", check.shape)
pd.DataFrame(cleaning_log)

Keep only required columns                    rows= 47,438  cols= 14  dropped identifiers, text, geometry columns, IssueDate and other leakage columns; kept 14 columns
Saved: (47438, 14)


,step,rows,cols,note
0,Load raw data,52199,20,
1,Drop duplicate rows / permit numbers,52199,20,removed 0
2,"Parse dates, drop unparseable",52199,22,
3,Remove negative processing days,52199,22,
4,Build target on_time,52199,23,median = 85 days; on-time share = 0.502
5,Project value: zero flag + log transform,52199,25,zero share = 0.152
6,Fill missing category/area/use; contractor flag,52199,26,
7,Group rare PropertyUse values,52199,28,21 levels kept
8,Time features + coordinates,52199,32,
9,Keep permits created 2017-2025,47438,32,removed pre-2017 and 2026
